# Scanner comparison: calibration of scanner 1 applied to scanner 2

Scanner 2 acquired a b=0 and a b=800 scan along each axis (X, Y, Z), repeated four times. The data are compared with
the calibration scanner (scanner 1) at the same gradient duration, using the registered noisy scans of both scanners
(`config.SCAN_TYPE`).

1. **Measured correction factor maps** of the two scanners (mean over the repeats, smoothed with a B-spline), after
   registering scanner 2 to scanner 1.
2. **Predicted and corrected maps** - the b-value maps of every scanner 2 scan are predicted from the duration-specific
   calibration maps of scanner 1 (`estimate_calibration.py`). For each axis the measured, predicted and corrected
   (measured / predicted) correction factor maps are shown with the distribution of the measured and corrected maps
   of each repeat.

In [ ]:
import config   # settings shared by all analysis scripts (also makes the root modules importable)

# General libraries
import numpy as np
import pandas as pd
import nibabel as nib
from functools import partial
from tqdm import tqdm
import ants

# Plotting libraries
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns
import colorcet as cc

# Custom functions
import registration as reg
import temperature as temp
import calib_utils as cal

config.PLOT_DIR.mkdir(exist_ok=True)

# Register the noisy scans of scanner 2 to scanner 1 before comparing (the SyN transform is estimated on the
# mean b=0 images). Set to False to compare the scans on their own grids (only the mask is registered).
USE_REGISTERED = True

### LOAD DATA

In [ ]:
def load_data(scans_fname, param_fname, mask_fname):
    """Load the scans (registered, config.SCAN_TYPE), their parameters and the mask of a dataset."""

    scans_nii = nib.load(scans_fname)
    scans_data = scans_nii.get_fdata().astype(np.float32)
    spacing = reg.spacing_from_affine(scans_nii.affine)
    mask = nib.load(mask_fname).get_fdata().astype(bool)

    df = pd.read_hdf(param_fname, key="df", mode="r")
    if scans_data.shape[-1] != df.shape[0]:
        raise ValueError(f"{scans_fname.name}: {scans_data.shape[-1]} scans but {df.shape[0]} rows in the parameter file")

    df["image"] = list(np.transpose(scans_data, [3,0,1,2]))
    df["adc"] = df["temperature"].apply(partial(temp.water_diffusivity, polynomial=True))

    return df, mask, spacing

In [ ]:
print(f"Scan type: {config.SCAN_TYPE}")

# Scanner 1 (calibration scans)
s1_df, s1_mask, s1_spacing = load_data(config.SCANS_FNAME, config.PARAM_FNAME, config.MASK_FNAME)

# Scanner 2
s2_df, s2_mask, s2_spacing = load_data(config.SCANNER2_SCANS_FNAME, config.SCANNER2_PARAM_FNAME, config.SCANNER2_MASK_FNAME)

# Scanner 2 uses a single gradient duration; compare with the scanner 1 scans of that duration
s2_durs = s2_df["dwi_grad_duration"].unique()
if len(s2_durs) != 1:
    raise ValueError(f"Expected one gradient duration in scanner 2, found {s2_durs}")
dur = float(s2_durs[0])
s1_relevant_df = s1_df[np.isclose(s1_df["dwi_grad_duration"], dur)]
print(f"Gradient duration: {dur * 1e3:.0f} ms. Scanner 1: {s1_relevant_df.shape[0]} scans, scanner 2: {s2_df.shape[0]} scans")

### REGISTER SCANNER 2 TO SCANNER 1

In [ ]:
# Mean b=0 images
s1_s0 = np.mean(s1_df[s1_df["b_value"] < config.B0_THRESHOLD]["image"].values, axis=0)
s2_s0 = np.mean(s2_df[s2_df["b_value"] < config.B0_THRESHOLD]["image"].values, axis=0)

fixed_a = ants.from_numpy(s1_s0, spacing=s1_spacing)
moving_a = ants.from_numpy(s2_s0, spacing=s2_spacing)

reg_tx = ants.registration(fixed=fixed_a,
                           moving=moving_a,
                           type_of_transform="SyN",
                           flow_sigma=10,
                           total_sigma=10,
                           aff_metric="mattes",
                           syn_metric="mattes",
                           grad_step=0.1)

# Apply the transform to every scan of scanner 2 and to its mask
s2_df["reg_image"] = [ants.apply_transforms(fixed=fixed_a,
                                            moving=ants.from_numpy(scan, spacing=s2_spacing),
                                            transformlist=reg_tx["fwdtransforms"],
                                            interpolator="cosineWindowedSinc").numpy().astype(np.float32)
                      for scan in tqdm(s2_df["image"].values, desc="Applying transform to scanner 2 scans")]

s2_mask_reg = ants.apply_transforms(fixed=fixed_a,
                                    moving=ants.from_numpy(s2_mask.astype(np.float32), spacing=s2_spacing),
                                    transformlist=reg_tx["fwdtransforms"],
                                    interpolator="nearestNeighbor").numpy().astype(bool)

# Region common to both scanners
marg = 5 # mm
combo_mask = cal.shrink_mask_mm(s1_mask, s1_spacing, marg) & cal.shrink_mask_mm(s2_mask_reg, s2_spacing, marg)

if USE_REGISTERED:
    s2_df["cmap_image"] = s2_df["reg_image"]
else:
    s2_df["cmap_image"] = s2_df["image"]
s1_relevant_df = s1_relevant_df.assign(cmap_image=s1_relevant_df["image"])

### MEASURED CORRECTION FACTOR MAPS

In [ ]:
def calc_cmaps_df(df, mask, image_col, b0_threshold=5):
    """
    Measured b-value map and correction factor map (c = b_measured / b_nominal) of every diffusion-weighted
    scan, using the preceding b=0 scan of the same axis.
    """

    _mask = mask.astype(bool)

    calib_parts = []

    for axis in tqdm(df["dwi_direction"].unique()):

        sub_df = df[df["dwi_direction"]==axis].copy()

        # Ensure scans are in acquisition order
        sub_df = sub_df.sort_values("scan_time").copy()

        # Put the b=0 image on b=0 rows, NaN elsewhere
        sub_df["s0_image"] = sub_df[image_col].where(sub_df["b_value"] < b0_threshold)

        # Forward-fill so each DWI scan gets the most recent preceding b=0 image
        sub_df["s0_image"] = sub_df["s0_image"].ffill()

        # Keep only diffusion-weighted scans
        dwi_df = sub_df[sub_df["b_value"] > b0_threshold].copy()

        # Drop rows that do not have a preceding b=0 scan
        dwi_df = dwi_df[dwi_df["s0_image"].notna()].copy()

        dwi_df["bmap"] = dwi_df.apply(
            lambda row: cal.bmap_3d(
                row["s0_image"],
                row[image_col],
                row["adc"],
                mask=_mask
            ),
            axis=1
        )
        dwi_df["cmap"] = dwi_df["bmap"] / dwi_df["b_value"]

        calib_parts.append(dwi_df)

    calib_df = pd.concat(calib_parts, axis=0, ignore_index=False)
    return calib_df

In [ ]:
s1_calib_df = calc_cmaps_df(s1_relevant_df, mask=combo_mask, image_col="cmap_image", b0_threshold=config.B0_THRESHOLD)
s2_calib_df = calc_cmaps_df(s2_df, mask=combo_mask, image_col="cmap_image", b0_threshold=config.B0_THRESHOLD)

print(s2_calib_df.groupby("seq_name").size().to_string())

In [ ]:
def fill_frame(array, crop_region):
    x1, x2, y1, y2 = list(crop_region)
    return array[x1:x2, y1:y2]

In [ ]:
# Scanner 2 is on the grid of scanner 1 once registered
s2_grid_spacing = s1_spacing if USE_REGISTERED else s2_spacing

comparison_seqs = sorted(s2_calib_df["seq_name"].unique())

n_row = 2
n_img_col = len(comparison_seqs)   # dynamic number of image columns
n_col = n_img_col + 1              # +1 for colorbar column

# -------------------------
# Layout
# -------------------------
fig = plt.figure(figsize=(2 * n_img_col + 1, 5))

gs = fig.add_gridspec(
    n_row,
    n_col,
    width_ratios=[1] * n_img_col + [0.1],
    wspace=0.25,
    hspace=0.35
)

axes = [
    [fig.add_subplot(gs[i, j]) for j in range(n_img_col)]
    for i in range(n_row)
]

cax = fig.add_subplot(gs[:, -1])

centre = np.array(combo_mask.shape) // 2
dz = -3

vmin = 0.65
vmax = 1.35
slice_crop = [2, -2, 2, -3]

fs_title = 12

for i, seq in enumerate(comparison_seqs):

    seq_cropped = seq.replace("_scanner_comp", "")

    s1_subset_df = s1_calib_df[
        s1_calib_df["seq_name"] == seq_cropped
    ].copy()

    s2_subset_df = s2_calib_df[
        s2_calib_df["seq_name"] == seq
    ].copy()

    n_min = min(s1_subset_df.shape[0], s2_subset_df.shape[0])

    s1_subset_df = s1_subset_df.head(n_min)
    s2_subset_df = s2_subset_df.head(n_min)

    s1_cmap = np.mean(s1_subset_df["cmap"].values)
    s2_cmap = np.mean(s2_subset_df["cmap"].values)

    s1_cmap_bspline = cal.bspline_estimation(
        s1_cmap,
        s1_spacing,
        mask=combo_mask,
        spline_param=config.SPLINE_PARAM
    )

    s2_cmap_bspline = cal.bspline_estimation(
        s2_cmap,
        s2_grid_spacing,
        mask=combo_mask,
        spline_param=config.SPLINE_PARAM
    )

    dw_axis = s1_subset_df["dwi_direction"].unique()[0]
    bval = int(np.round(s1_subset_df["b_value"].unique()[0], -2))

    # -------------------------
    # Slices
    # -------------------------
    s1_slice = s1_cmap_bspline[:, :, centre[2] + dz]
    s1_slice = fill_frame(np.rot90(s1_slice, k=1), slice_crop)

    s2_slice = s2_cmap_bspline[:, :, centre[2] + dz]
    s2_slice = fill_frame(np.rot90(s2_slice, k=1), slice_crop)

    mask_slice = combo_mask[:, :, centre[2] + dz]

    plot_mask = np.full_like(mask_slice, np.nan, dtype=float)
    plot_mask[mask_slice == 0] = 1
    plot_mask = fill_frame(np.rot90(plot_mask, k=1), slice_crop)

    # -------------------------
    # Plotting
    # -------------------------
    ax0 = axes[0][i]
    ax1 = axes[1][i]

    ax0.imshow(
        s1_slice,
        vmin=vmin,
        vmax=vmax,
        cmap=cc.cm.coolwarm,
        interpolation="nearest"
    )

    ax0.imshow(
        plot_mask,
        vmin=0,
        vmax=1,
        cmap=cm.Greys_r,
        interpolation="nearest"
    )

    ax0.set_title(
        rf"$b_{{{dw_axis}}} = {bval}$",
        fontsize=fs_title
    )

    im = ax1.imshow(
        s2_slice,
        vmin=vmin,
        vmax=vmax,
        cmap=cc.cm.coolwarm,
        interpolation="nearest"
    )

    ax1.imshow(
        plot_mask,
        vmin=0,
        vmax=1,
        cmap=cm.Greys_r,
        interpolation="nearest"
    )

# -------------------------
# Row labels
# -------------------------
for i in range(n_row):

    axes[i][0].text(
        -0.05,
        0.5,
        rf"$Scanner\ {i+1}$",
        transform=axes[i][0].transAxes,
        fontsize=fs_title,
        ha="right",
        va="center",
    )

# -------------------------
# Turn axes off
# -------------------------
for row in axes:
    for ax in row:
        ax.set_axis_off()

# -------------------------
# Colorbar
# -------------------------
cbar = fig.colorbar(im, cax=cax)

cbar.ax.set_title(
    r"$c_{Meas}$",
    fontsize=12,
    pad=10
)

cbar.ax.tick_params(labelsize=12)

plot_fname = config.PLOT_DIR / f"scanner_comparison_measured_b_{bval:.0f}.png"

fig.savefig(
    fname=plot_fname,
    dpi=600,
    format="png",
    bbox_inches="tight",
    pad_inches=0.1)
plt.show()

### PREDICTED AND CORRECTED MAPS

The b-value map of every scanner 2 scan is predicted from the duration-specific calibration maps of scanner 1 (the
same maps that are applied to the phantom and in vivo data, `config.a_dur_fname`). Unlike the leave-one-out figures, the
scanner 2 scans were not used to fit the calibration, so all repeats are predicted by the same maps.

In [ ]:
# Calibration maps of this gradient duration (scanner 1)
A, A_mask, sigma, A_spacing = cal.unpack_A_nii(config.a_dur_fname(dur))
if A_mask.shape != combo_mask.shape:
    raise ValueError(f"Calibration maps {A_mask.shape} and scanner 2 data {combo_mask.shape} are on different grids")

# Region evaluated: common to both scanners and covered by the calibration maps
eval_mask = combo_mask & A_mask

axis_order = ["X", "Y", "Z"]
n_axes = len(axis_order)
n_repeats = s2_calib_df.groupby("dwi_direction").size().min()
b_values = {axis: s2_calib_df[s2_calib_df["dwi_direction"] == axis]["b_value"].mean() for axis in axis_order}

# Stored as (X, Y, Z, axis, repeat)
measured_all = np.zeros((*eval_mask.shape, n_axes, n_repeats), dtype=np.float32)
predicted_all = np.zeros_like(measured_all)
corrected_all = np.zeros_like(measured_all)

for i_axis, axis in enumerate(axis_order):

    axis_df = s2_calib_df[s2_calib_df["dwi_direction"] == axis].sort_values("scan_time").head(n_repeats)

    for i_rep, (_, row) in enumerate(axis_df.iterrows()):

        pred_bmap = cal.est_bmap(row["grad_vec"],
                                 delta_small=row["dwi_grad_duration"],
                                 delta_big=row["TE"] / 2,
                                 A=A,
                                 sigma=sigma,
                                 mask=A_mask)

        eval_cmap = row["cmap"].astype(np.float32)
        pred_cmap = (pred_bmap / row["b_value"]).astype(np.float32)

        valid = (eval_cmap > 0) & (pred_cmap > 0) & eval_mask
        corr_cmap = np.zeros_like(eval_cmap)
        corr_cmap[valid] = eval_cmap[valid] / pred_cmap[valid]

        measured_all[..., i_axis, i_rep] = eval_cmap
        predicted_all[..., i_axis, i_rep] = pred_cmap
        corrected_all[..., i_axis, i_rep] = corr_cmap

# Average across the repeats for imshow
mean_measured = np.mean(measured_all, axis=-1)
mean_predicted = np.mean(predicted_all, axis=-1)
mean_corrected = np.mean(corrected_all, axis=-1)

In [ ]:
i_slice = mean_measured.shape[2] // 2
lim = 0.5

# -------------------------
# Consistent font settings
# -------------------------
fs_seq_title = 20
fs_im_title = 12
fs_ticks = 12
fs_legend = 9
fs_cbar = 14
kde_lw = 0.8

# -------------------------
# ONE row: a column per axis, each with the three maps above the distributions
# -------------------------
fig = plt.figure(figsize=(6.25 * n_axes, 4.5))

outer_gs = fig.add_gridspec(1, n_axes, wspace=0.25)

image_axes = []
last_im = None

for j, axis in enumerate(axis_order):

    inner_gs = outer_gs[0, j].subgridspec(2, 3, hspace=0.15, wspace=0.05)

    ax1 = fig.add_subplot(inner_gs[0, 0])
    ax2 = fig.add_subplot(inner_gs[0, 1])
    ax3 = fig.add_subplot(inner_gs[0, 2])
    ax4 = fig.add_subplot(inner_gs[1, :])

    meas = np.rot90(mean_measured[:, :, i_slice, j], k=1)[:, :-1]
    pred = np.rot90(mean_predicted[:, :, i_slice, j], k=1)[:, :-1]
    corr = np.rot90(mean_corrected[:, :, i_slice, j], k=1)[:, :-1]

    mask_slice = np.rot90(eval_mask[:, :, i_slice], k=1)[:, :-1]
    plot_mask = np.full_like(meas, np.nan, dtype=float)
    plot_mask[mask_slice == 0] = 1

    for ax, data, lab in zip([ax1, ax2, ax3], [meas, pred, corr], ["Measured", "Predicted", "Corrected"]):
        im = ax.imshow(
            data,
            vmin=1 - lim,
            vmax=1 + lim,
            cmap=cc.cm.coolwarm,
            interpolation=None
        )

        ax.imshow(
            plot_mask,
            vmin=0,
            vmax=1,
            cmap="Greys_r",
            interpolation="nearest"
        )

        ax.set_axis_off()

        ax.text(
            0.5, -0.0, lab,
            transform=ax.transAxes,
            ha="center",
            va="top",
            fontsize=fs_im_title
        )

        image_axes.append(ax)
        last_im = im

    # -------------------------
    # KDE plots: one curve per repeat
    # -------------------------
    for i in range(n_repeats):
        in_region = eval_mask & (corrected_all[..., j, i] > 0)   # same voxels for measured and corrected

        sns.kdeplot(
            measured_all[..., j, i][in_region],
            color="darkorange",
            clip=(1 - lim, 1 + lim),
            ax=ax4,
            lw=kde_lw
        )

        sns.kdeplot(
            corrected_all[..., j, i][in_region],
            color="cornflowerblue",
            clip=(1 - lim, 1 + lim),
            ax=ax4,
            lw=kde_lw
        )

    meas_patch = mpatches.Patch(facecolor="darkorange", edgecolor="k", linewidth=0.5, label="Measured")
    corr_patch = mpatches.Patch(facecolor="cornflowerblue", edgecolor="k", linewidth=0.5, label="Corrected")

    ax4.set_xlim((1 - lim, 1 + lim))
    ax4.set_ylim((0, 10))
    ax4.grid(alpha=0.25)
    ax4.tick_params(labelsize=fs_ticks)
    ax4.legend(handles=[meas_patch, corr_patch], fontsize=fs_legend, frameon=False, loc="upper left")
    ax4.set_xlabel(r"$c(\mathbf{r})$", fontsize=fs_im_title)
    ax4.spines["top"].set_visible(False)
    ax4.spines["right"].set_visible(False)

    # Column title: the axis
    ax2.set_title(axis, fontsize=fs_seq_title, fontstyle="italic", pad=10)

# -------------------------
# Colorbar
# -------------------------
x1 = max(ax.get_position().x1 for ax in image_axes)
y0 = min(ax.get_position().y0 for ax in image_axes)
y1 = max(ax.get_position().y1 for ax in image_axes)

cax = fig.add_axes([x1 + 0.01, y0, 0.004, y1 - y0])
cbar = fig.colorbar(last_im, cax=cax)
cbar.ax.set_title(r"$c(r)$", fontsize=fs_cbar, pad=10)
cbar.ax.tick_params(labelsize=fs_ticks)

# Dashed separators between the columns
for j in range(n_axes - 1):
    left = outer_gs[0, j].get_position(fig)
    right = outer_gs[0, j + 1].get_position(fig)
    x = (left.x1 + right.x0) / 2
    fig.add_artist(Line2D([x, x], [left.y0 - 0.02, left.y1 + 0.02], transform=fig.transFigure,
                          linestyle="--", linewidth=1.2, color="black", alpha=0.5))

# -------------------------
# Single suptitle
# -------------------------
mean_b = int(np.round(np.mean(list(b_values.values())), -2))
plt.suptitle(rf"$\delta = {dur * 1e3:.0f}\ \mathrm{{ms}},\ b = {mean_b}$", fontsize=fs_seq_title, y=1.08)

# -------------------------
# Save
# -------------------------
plot_fname = config.PLOT_DIR / f"scanner_comparison_corrected_dur_{dur * 1e3:.0f}.png"
fig.savefig(fname=plot_fname, dpi=600, format="png", bbox_inches="tight", pad_inches=0.025)
plt.show()

#### Summary of the distributions

In [ ]:
# Spread of the correction factor over the evaluated region, with and without the calibration of scanner 1.
# "Per scan" is the spread within a single repeat (noise limited); "mean map" is the spread of the mean over
# the repeats, which is dominated by systematic (spatial) variation.
rows = []
for j, axis in enumerate(axis_order):
    for name, maps in [("Measured", measured_all), ("Corrected", corrected_all)]:
        in_region = eval_mask & np.all(corrected_all[..., j, :] > 0, axis=-1)
        per_scan = np.stack([maps[..., j, i][in_region] for i in range(n_repeats)])
        mean_map = per_scan.mean(axis=0)
        rows.append({"axis": axis,
                     "map": name,
                     "median": np.median(per_scan),
                     "std per scan": per_scan.std(axis=1).mean(),
                     "std of mean map": mean_map.std(),
                     "IQR of mean map": np.subtract(*np.percentile(mean_map, [75, 25]))})
summary_df = pd.DataFrame(rows)
display(summary_df.round(4))

summary_df.to_csv(config.PLOT_DIR / "scanner_comparison_summary.csv", index=False)